# CS13 TRN100k Point-Cloud and Surface Reconstruction

Construct the full CS13 point cloud, reconstruct a surface from a 100,000-cell TRN subset, and review the point-cloud/mesh overlay.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import pyvista as pv

pv.global_theme.transparent_background = True
import spateo as st


In [ ]:
cpo = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]


# v4


## Load and validate data


In [ ]:
# pc model reconstruction
cs13 = st.read_h5ad("/DATA/User/gaomohan/DATA/CS13_Project/cs13/CS13.final.h5ad")
# mesh model reconstruction
cs13_100k = st.read_h5ad("/DATA/User/gaomohan/DATA/CS13_Project/cs13/cs13_trn_100k_v4.h5ad")
cs13, cs13_100k


## Construct the point-cloud model


In [ ]:
cs13_pc, plot_cmap = st.tdr.construct_pc(
    adata=cs13.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)

cs13_100k_pc, plot_cmap = st.tdr.construct_pc(
    adata=cs13_100k.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


# PC model


In [ ]:
# Enable transparent backgrounds.
pv.global_theme.transparent_background = True

st.pl.three_d_plot(
    model=cs13_pc,
    key="tissue",
    model_style="points",
    model_size=1.2,
    opacity=0.8,
    ambient=0.6,
    colormap="#8F8D8D",
    background="white",  # White supports antialiasing; the exported PNG still has alpha=0.
    show_axes=True,
    show_legend=False,
    cpo=cpo,
    window_size=(1200, 1200),
    # Export to a PNG format that supports an alpha channel.
    off_screen=True,
    jupyter="static",
)


# 100k pc -> 100k mesh


In [ ]:
# Enable transparent backgrounds.
pv.global_theme.transparent_background = True
st.pl.three_d_plot(
    model=cs13_100k_pc,
    key="tissue",
    model_style="points",
    model_size=1.2,
    opacity=0.8,
    ambient=0.6,
    colormap="#8F8D8D",
    background="white",  # White supports antialiasing; the exported PNG still has alpha=0.
    show_axes=False,
    show_legend=False,
    cpo=cpo,
    window_size=(1200, 1200),
    # Export to a PNG format that supports an alpha channel.
    off_screen=True,
    jupyter="static",
)


## Reconstruct the surface mesh


In [ ]:
cs13_100k_mesh, _, _ = st.tdr.construct_surface(
    pc=cs13_100k_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.24},
    smooth=5500,
    scale_factor=1.04,
)


In [ ]:
st.pl.three_d_plot(
    model=cs13_100k_mesh,
    key="tissue",
    model_style="surface",
    show_axes=False,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo,
)


In [ ]:
cs13_inside_mesh = st.read_h5ad(
    "/DATA/User/gaomohan/DATA/CS13_Project/cs13/CS13.final.inside_cs13_100k_mesh.h5ad"
)
cs13_inside_mesh


In [ ]:
cs13_inside_pc, plot_cmap = st.tdr.construct_pc(
    adata=cs13_inside_mesh.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([cs13_100k_mesh, cs13_inside_pc]),
    # Both models use the `tissue` label.
    key=["tissue", "tissue"],
    model_style=["surface", "points"],
    # Point-cloud point size.
    model_size=1,
    # Keep the mesh translucent so points remain visible.
    opacity=[0.6, 0.7],
    # Ambient-light contribution.
    ambient=0.4,
    # Use light grey for the mesh and dark grey for the points.
    colormap=[
        "#D9D9D9",  # mesh
        "#8F8D8D",  # point cloud
    ],
    background="white",
    show_axes=False,
    show_legend=False,
    cpo=cpo,
    window_size=(1200, 1200),
    off_screen=True,
    jupyter="static",
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([cs13_100k_mesh, cs13_100k_pc]),
    # Both models use the `tissue` label.
    key=["tissue", "tissue"],
    model_style=["surface", "points"],
    # Point-cloud point size.
    model_size=1,
    # Keep the mesh translucent so points remain visible.
    opacity=[0.6, 0.7],
    # Ambient-light contribution.
    ambient=0.4,
    # Use light grey for the mesh and dark grey for the points.
    colormap=[
        "#D9D9D9",  # mesh
        "#8F8D8D",  # point cloud
    ],
    background="white",
    show_axes=False,
    show_legend=False,
    cpo=cpo,
    window_size=(1200, 1200),
    off_screen=True,
    jupyter="static",
)


In [ ]:
st.tdr.save_model(model=cs13_pc, filename="cs13_pc.vtk")
st.tdr.save_model(model=cs13_100k_mesh, filename="cs13_100k_mesh.vtk")
